# Deep Learning Mastery with d2l.ai 🧠

Welcome to your comprehensive deep learning journey! This notebook serves as your practical guide to mastering deep learning concepts using d2l.ai (Dive into Deep Learning) and other excellent resources.

## 📖 Learning Objectives

By the end of this notebook, you will:
- Understand the fundamental concepts of deep learning
- Implement neural networks from scratch and using frameworks
- Build CNNs for computer vision tasks
- Create RNNs and Transformers for sequence modeling
- Apply advanced optimization and regularization techniques
- Master the practical aspects of training and evaluating deep learning models

## 📚 Resources Used

- **Primary**: [d2l.ai](https://d2l.ai/) - Interactive deep learning book
- **Frameworks**: PyTorch, TensorFlow
- **Visualization**: Matplotlib, Seaborn, Plotly
- **Data**: Various datasets for hands-on practice

Let's begin this exciting journey into the world of deep learning!

## 1. Environment Setup and Library Installation 🔧

First, let's ensure we have all the necessary libraries installed. If you haven't set up your environment yet, please refer to `environments/setup.md` in the repository root.

### Installation Commands
```bash
# Install d2l and core packages
pip install d2l torch torchvision matplotlib numpy pandas scikit-learn

# For conda users
conda install pytorch torchvision -c pytorch
pip install d2l
```

In [ ]:
# Check if we're in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    # Install packages in Colab
    !pip install d2l
except:
    IN_COLAB = False
    print("Running locally")

# Verify installations
try:
    import d2l
    print(f"✅ d2l version: {d2l.__version__}")
except ImportError:
    print("❌ d2l not found. Please install with: pip install d2l")

try:
    import torch
    print(f"✅ PyTorch version: {torch.__version__}")
    print(f"✅ CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"✅ CUDA version: {torch.version.cuda}")
        print(f"✅ GPU count: {torch.cuda.device_count()}")
except ImportError:
    print("❌ PyTorch not found. Please install PyTorch")

print("\n🎉 Environment check complete!")

## 2. Import Essential Deep Learning Libraries 📚

Let's import all the essential libraries we'll need for our deep learning journey.

In [ ]:
# Core deep learning frameworks
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset

# d2l.ai utilities
import d2l.torch as d2l

# Scientific computing
import numpy as np
import pandas as pd
import scipy.stats as stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

# Utilities
import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Set up plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("🚀 All libraries imported successfully!")
print(f"📱 Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 3. Data Loading and Preprocessing 📊

Understanding data is fundamental to deep learning. Let's explore data loading, preprocessing, and visualization techniques.

In [ ]:
# Create synthetic datasets for learning
def create_synthetic_data(n_samples=1000, n_features=20, noise=0.1):
    """Create synthetic datasets for different types of learning problems"""
    
    # 1. Regression dataset
    X_reg, y_reg = make_regression(
        n_samples=n_samples, 
        n_features=n_features, 
        noise=noise,
        random_state=42
    )
    
    # 2. Classification dataset  
    X_clf, y_clf = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_classes=2,
        n_redundant=0,
        random_state=42
    )
    
    return (X_reg, y_reg), (X_clf, y_clf)

# Generate data
(X_reg, y_reg), (X_clf, y_clf) = create_synthetic_data()

print(f"📈 Regression data shape: X={X_reg.shape}, y={y_reg.shape}")
print(f"🎯 Classification data shape: X={X_clf.shape}, y={y_clf.shape}")

# Data preprocessing
def preprocess_data(X, y, test_size=0.2):
    """Preprocess data: split and normalize"""
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

# Preprocess both datasets
X_reg_train, X_reg_test, y_reg_train, y_reg_test, reg_scaler = preprocess_data(X_reg, y_reg)
X_clf_train, X_clf_test, y_clf_train, y_clf_test, clf_scaler = preprocess_data(X_clf, y_clf)

print(f"✅ Data preprocessing complete!")
print(f"Training set sizes: Regression={X_reg_train.shape[0]}, Classification={X_clf_train.shape[0]}")

# Visualize data distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Regression target distribution
axes[0, 0].hist(y_reg_train, bins=30, alpha=0.7, color='blue')
axes[0, 0].set_title('Regression Target Distribution')
axes[0, 0].set_xlabel('Target Value')
axes[0, 0].set_ylabel('Frequency')

# Classification target distribution
unique, counts = np.unique(y_clf_train, return_counts=True)
axes[0, 1].bar(unique, counts, alpha=0.7, color=['red', 'green'])
axes[0, 1].set_title('Classification Target Distribution')
axes[0, 1].set_xlabel('Class')
axes[0, 1].set_ylabel('Count')

# Feature correlations (first 5 features)
reg_corr = np.corrcoef(X_reg_train[:, :5].T)
im1 = axes[1, 0].imshow(reg_corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1, 0].set_title('Feature Correlations (Regression)')
plt.colorbar(im1, ax=axes[1, 0])

clf_corr = np.corrcoef(X_clf_train[:, :5].T)
im2 = axes[1, 1].imshow(clf_corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1, 1].set_title('Feature Correlations (Classification)')
plt.colorbar(im2, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print("📊 Data visualization complete!")

## 4. Linear Regression Implementation 📏

Linear regression is the foundation of many machine learning algorithms. Let's implement it from scratch to understand gradient descent and loss functions.

**Key Concepts:**
- **Gradient Descent**: Optimization algorithm to minimize loss
- **Loss Function**: Measures prediction error (MSE for regression)
- **Learning Rate**: Controls step size in gradient descent
- **Feature Matrix**: Input features organized in matrix form

In [ ]:
class LinearRegressionFromScratch:
    """Linear regression implementation from scratch using gradient descent"""
    
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.tolerance = tolerance
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        """Train the linear regression model"""
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.random.normal(0, 0.01, n_features)
        self.bias = 0
        
        # Gradient descent
        for i in range(self.max_iterations):
            # Forward pass
            predictions = self.predict(X)
            
            # Compute loss (MSE)
            loss = np.mean((predictions - y) ** 2)
            self.loss_history.append(loss)
            
            # Compute gradients
            dw = (2 / n_samples) * X.T @ (predictions - y)
            db = (2 / n_samples) * np.sum(predictions - y)
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Check convergence
            if i > 0 and abs(self.loss_history[-2] - self.loss_history[-1]) < self.tolerance:
                print(f"Converged after {i+1} iterations")
                break
    
    def predict(self, X):
        """Make predictions"""
        return X @ self.weights + self.bias
    
    def score(self, X, y):
        """Calculate R² score"""
        predictions = self.predict(X)
        ss_res = np.sum((y - predictions) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

# Train from-scratch model
print("🔧 Training Linear Regression from Scratch...")
model_scratch = LinearRegressionFromScratch(learning_rate=0.01, max_iterations=1000)
model_scratch.fit(X_reg_train, y_reg_train)

# Make predictions
y_pred_scratch = model_scratch.predict(X_reg_test)
mse_scratch = mean_squared_error(y_reg_test, y_pred_scratch)
r2_scratch = model_scratch.score(X_reg_test, y_reg_test)

print(f"✅ From Scratch - MSE: {mse_scratch:.4f}, R²: {r2_scratch:.4f}")

# Compare with PyTorch implementation
class LinearRegressionPyTorch(nn.Module):
    def __init__(self, input_size):
        super(LinearRegressionPyTorch, self).__init__()
        self.linear = nn.Linear(input_size, 1)
    
    def forward(self, x):
        return self.linear(x).squeeze()

# Convert data to tensors
X_train_tensor = torch.FloatTensor(X_reg_train)
y_train_tensor = torch.FloatTensor(y_reg_train)
X_test_tensor = torch.FloatTensor(X_reg_test)
y_test_tensor = torch.FloatTensor(y_reg_test)

# Train PyTorch model
print("🔥 Training Linear Regression with PyTorch...")
model_pytorch = LinearRegressionPyTorch(X_reg_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model_pytorch.parameters(), lr=0.01)

pytorch_losses = []
for epoch in range(1000):
    # Forward pass
    outputs = model_pytorch(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    pytorch_losses.append(loss.item())
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f'Epoch [{epoch}/1000], Loss: {loss.item():.4f}')

# Evaluate PyTorch model
with torch.no_grad():
    y_pred_pytorch = model_pytorch(X_test_tensor).numpy()
    mse_pytorch = mean_squared_error(y_reg_test, y_pred_pytorch)
    r2_pytorch = 1 - (np.sum((y_reg_test - y_pred_pytorch) ** 2) / 
                     np.sum((y_reg_test - np.mean(y_reg_test)) ** 2))

print(f"✅ PyTorch - MSE: {mse_pytorch:.4f}, R²: {r2_pytorch:.4f}")

# Visualize training progress
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(model_scratch.loss_history, label='From Scratch', color='blue')
plt.plot(pytorch_losses, label='PyTorch', color='red', alpha=0.7)
plt.title('Training Loss Comparison')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.legend()
plt.yscale('log')

plt.subplot(1, 3, 2)
plt.scatter(y_reg_test, y_pred_scratch, alpha=0.6, label='From Scratch', color='blue')
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('From Scratch: Predictions vs True')
plt.legend()

plt.subplot(1, 3, 3)
plt.scatter(y_reg_test, y_pred_pytorch, alpha=0.6, label='PyTorch', color='red')
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('PyTorch: Predictions vs True')
plt.legend()

plt.tight_layout()
plt.show()

print("📈 Linear regression comparison complete!")

## 5. Neural Network from Scratch 🧠

Now let's build a complete neural network from scratch to understand:
- **Forward Propagation**: How data flows through the network
- **Backpropagation**: How gradients flow backwards to update weights  
- **Activation Functions**: Non-linear transformations (ReLU, Sigmoid, etc.)
- **Loss Functions**: How we measure prediction quality

In [ ]:
class NeuralNetworkFromScratch:
    """Multi-layer neural network implemented from scratch"""
    
    def __init__(self, layer_sizes, learning_rate=0.01):
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.num_layers = len(layer_sizes)
        self.loss_history = []
        
        # Initialize weights and biases
        self.weights = {}
        self.biases = {}
        
        for i in range(1, self.num_layers):
            # Xavier initialization
            self.weights[i] = np.random.randn(layer_sizes[i-1], layer_sizes[i]) * np.sqrt(2.0 / layer_sizes[i-1])
            self.biases[i] = np.zeros((1, layer_sizes[i]))
    
    def relu(self, x):
        """ReLU activation function"""
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        """Derivative of ReLU"""
        return (x > 0).astype(float)
    
    def sigmoid(self, x):
        """Sigmoid activation function"""
        return 1 / (1 + np.exp(-np.clip(x, -250, 250)))  # Clip to prevent overflow
    
    def sigmoid_derivative(self, x):
        """Derivative of sigmoid"""
        s = self.sigmoid(x)
        return s * (1 - s)
    
    def forward_propagation(self, X):
        """Forward propagation through the network"""
        self.activations = {0: X}
        self.z_values = {}
        
        for i in range(1, self.num_layers):
            # Linear transformation
            self.z_values[i] = self.activations[i-1] @ self.weights[i] + self.biases[i]
            
            # Activation function
            if i == self.num_layers - 1:  # Output layer
                if self.layer_sizes[-1] == 1:  # Regression or binary classification
                    self.activations[i] = self.z_values[i]  # Linear for regression
                else:  # Multi-class classification
                    self.activations[i] = self.sigmoid(self.z_values[i])
            else:  # Hidden layers
                self.activations[i] = self.relu(self.z_values[i])
        
        return self.activations[self.num_layers - 1]
    
    def backward_propagation(self, X, y):
        """Backward propagation to compute gradients"""
        m = X.shape[0]  # Number of samples
        
        # Initialize gradients
        dW = {}\n        db = {}\n        \n        # Output layer error\n        if self.layer_sizes[-1] == 1:  # Regression\n            delta = self.activations[self.num_layers - 1] - y.reshape(-1, 1)\n        else:  # Classification\n            delta = self.activations[self.num_layers - 1] - y\n        \n        # Backward pass\n        for i in range(self.num_layers - 1, 0, -1):\n            # Compute gradients\n            dW[i] = (1/m) * self.activations[i-1].T @ delta\n            db[i] = (1/m) * np.sum(delta, axis=0, keepdims=True)\n            \n            # Propagate error to previous layer (if not input layer)\n            if i > 1:\n                delta = (delta @ self.weights[i].T) * self.relu_derivative(self.z_values[i-1])\n        \n        return dW, db\n    \n    def update_parameters(self, dW, db):\n        """Update weights and biases using gradients"""\n        for i in range(1, self.num_layers):\n            self.weights[i] -= self.learning_rate * dW[i]\n            self.biases[i] -= self.learning_rate * db[i]\n    \n    def compute_loss(self, y_true, y_pred):\n        """Compute loss (MSE for regression, cross-entropy for classification)"""\n        if self.layer_sizes[-1] == 1:  # Regression\n            return np.mean((y_true.reshape(-1, 1) - y_pred) ** 2)\n        else:  # Binary classification\n            # Cross-entropy loss\n            y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)  # Prevent log(0)\n            return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))\n    \n    def fit(self, X, y, epochs=1000, verbose=True):\n        """Train the neural network"""\n        for epoch in range(epochs):\n            # Forward propagation\n            y_pred = self.forward_propagation(X)\n            \n            # Compute loss\n            loss = self.compute_loss(y, y_pred)\n            self.loss_history.append(loss)\n            \n            # Backward propagation\n            dW, db = self.backward_propagation(X, y)\n            \n            # Update parameters\n            self.update_parameters(dW, db)\n            \n            # Print progress\n            if verbose and epoch % 100 == 0:\n                print(f'Epoch {epoch}, Loss: {loss:.4f}')\n    \n    def predict(self, X):\n        """Make predictions"""\n        return self.forward_propagation(X)\n\n# Test neural network on classification problem\nprint("🧠 Training Neural Network from Scratch on Classification...")\n\n# Prepare classification data\nX_clf_nn = X_clf_train\ny_clf_nn = y_clf_train.reshape(-1, 1)\n\n# Create and train network\nnn_classifier = NeuralNetworkFromScratch(\n    layer_sizes=[X_clf_nn.shape[1], 64, 32, 1],  # Input -> 64 -> 32 -> 1 output\n    learning_rate=0.01\n)\n\nnn_classifier.fit(X_clf_nn, y_clf_nn, epochs=1000, verbose=True)\n\n# Make predictions\ny_pred_nn = nn_classifier.predict(X_clf_test)\ny_pred_binary = (y_pred_nn > 0.5).astype(int).flatten()\n\n# Calculate accuracy\naccuracy_nn = accuracy_score(y_clf_test, y_pred_binary)\nprint(f"\\n✅ Neural Network Accuracy: {accuracy_nn:.4f}")

# Test neural network on regression problem
print("\\n📈 Training Neural Network from Scratch on Regression...")

nn_regressor = NeuralNetworkFromScratch(
    layer_sizes=[X_reg_train.shape[1], 64, 32, 1],  # Input -> 64 -> 32 -> 1 output
    learning_rate=0.001
)

nn_regressor.fit(X_reg_train, y_reg_train, epochs=1000, verbose=True)

# Make predictions
y_pred_nn_reg = nn_regressor.predict(X_reg_test)
mse_nn = mean_squared_error(y_reg_test, y_pred_nn_reg.flatten())
print(f"\\n✅ Neural Network MSE: {mse_nn:.4f}")

# Visualize training progress and results
plt.figure(figsize=(15, 5))

# Classification loss
plt.subplot(1, 3, 1)
plt.plot(nn_classifier.loss_history)
plt.title('Classification Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.yscale('log')

# Regression loss  
plt.subplot(1, 3, 2)
plt.plot(nn_regressor.loss_history)
plt.title('Regression Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.yscale('log')

# Regression predictions
plt.subplot(1, 3, 3)
plt.scatter(y_reg_test, y_pred_nn_reg, alpha=0.6)
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('NN Regression: Predictions vs True')

plt.tight_layout()
plt.show()

print("🎉 Neural network from scratch implementation complete!")

## 6. Using Deep Learning Frameworks 🔥

While understanding the fundamentals is crucial, modern deep learning relies heavily on frameworks like PyTorch and TensorFlow. Let's see how the same networks can be built much more efficiently!

In [ ]:
# PyTorch Neural Network
class PyTorchNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, task='classification'):
        super(PyTorchNN, self).__init__()
        self.task = task
        
        # Build layers dynamically
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))  # Add regularization
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, output_size))
        
        # Add final activation for classification
        if task == 'classification' and output_size == 1:
            layers.append(nn.Sigmoid())
        elif task == 'classification' and output_size > 1:
            layers.append(nn.Softmax(dim=1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

def train_pytorch_model(model, X_train, y_train, X_test, y_test, task='classification', epochs=1000):
    """Train PyTorch model with proper setup"""
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train)
    X_test_tensor = torch.FloatTensor(X_test)
    y_test_tensor = torch.FloatTensor(y_test)
    
    # Setup loss and optimizer
    if task == 'classification':
        if len(y_train.shape) == 1 or y_train.shape[1] == 1:
            criterion = nn.BCELoss()\n            y_train_tensor = y_train_tensor.float()\n        else:\n            criterion = nn.CrossEntropyLoss()\n    else:\n        criterion = nn.MSELoss()\n    \n    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)\n    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=50, factor=0.5)\n    \n    # Training loop\n    train_losses = []\n    val_losses = []\n    \n    for epoch in range(epochs):\n        # Training\n        model.train()\n        optimizer.zero_grad()\n        \n        outputs = model(X_train_tensor)\n        if task == 'regression':\n            outputs = outputs.squeeze()\n        \n        loss = criterion(outputs, y_train_tensor)\n        loss.backward()\n        optimizer.step()\n        \n        train_losses.append(loss.item())\n        \n        # Validation\n        model.eval()\n        with torch.no_grad():\n            val_outputs = model(X_test_tensor)\n            if task == 'regression':\n                val_outputs = val_outputs.squeeze()\n            val_loss = criterion(val_outputs, y_test_tensor)\n            val_losses.append(val_loss.item())\n        \n        scheduler.step(val_loss)\n        \n        if epoch % 100 == 0:\n            print(f'Epoch [{epoch}/{epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}')\n    \n    return train_losses, val_losses\n\n# Train classification model with PyTorch\nprint("🔥 Training Classification with PyTorch...")\npytorch_clf = PyTorchNN(\n    input_size=X_clf_train.shape[1],\n    hidden_sizes=[64, 32],\n    output_size=1,\n    task='classification'\n)\n\ntrain_losses_clf, val_losses_clf = train_pytorch_model(\n    pytorch_clf, X_clf_train, y_clf_train, X_clf_test, y_clf_test, 'classification'\n)\n\n# Evaluate classification\npytorch_clf.eval()\nwith torch.no_grad():\n    test_outputs = pytorch_clf(torch.FloatTensor(X_clf_test))\n    test_predictions = (test_outputs.numpy() > 0.5).astype(int).flatten()\n    pytorch_clf_accuracy = accuracy_score(y_clf_test, test_predictions)\n\nprint(f"✅ PyTorch Classification Accuracy: {pytorch_clf_accuracy:.4f}")\n\n# Train regression model with PyTorch\nprint("\\n📈 Training Regression with PyTorch...")\npytorch_reg = PyTorchNN(\n    input_size=X_reg_train.shape[1],\n    hidden_sizes=[64, 32],\n    output_size=1,\n    task='regression'\n)\n\ntrain_losses_reg, val_losses_reg = train_pytorch_model(\n    pytorch_reg, X_reg_train, y_reg_train, X_reg_test, y_reg_test, 'regression'\n)\n\n# Evaluate regression\npytorch_reg.eval()\nwith torch.no_grad():\n    test_outputs_reg = pytorch_reg(torch.FloatTensor(X_reg_test))\n    pytorch_reg_mse = mean_squared_error(y_reg_test, test_outputs_reg.numpy())\n\nprint(f"✅ PyTorch Regression MSE: {pytorch_reg_mse:.4f}")\n\n# Compare framework vs from-scratch\nprint("\\n📊 Comparison Summary:")
print("="*50)
print("CLASSIFICATION RESULTS:")
print(f"From Scratch:  {accuracy_nn:.4f}")
print(f"PyTorch:       {pytorch_clf_accuracy:.4f}")
print("\\nREGRESSION RESULTS (MSE):")
print(f"From Scratch:  {mse_nn:.4f}")
print(f"PyTorch:       {pytorch_reg_mse:.4f}")

# Visualize training curves
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.plot(train_losses_clf, label='Train', alpha=0.7)
plt.plot(val_losses_clf, label='Validation', alpha=0.7)
plt.title('PyTorch Classification Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.yscale('log')

plt.subplot(2, 3, 2)
plt.plot(train_losses_reg, label='Train', alpha=0.7)
plt.plot(val_losses_reg, label='Validation', alpha=0.7)
plt.title('PyTorch Regression Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.yscale('log')

plt.subplot(2, 3, 3)
plt.plot(nn_classifier.loss_history, label='From Scratch', alpha=0.7)
plt.plot(train_losses_clf, label='PyTorch', alpha=0.7)
plt.title('Classification: Framework Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.yscale('log')

plt.subplot(2, 3, 4)
frameworks = ['From Scratch', 'PyTorch']
clf_scores = [accuracy_nn, pytorch_clf_accuracy]
plt.bar(frameworks, clf_scores, color=['blue', 'orange'], alpha=0.7)
plt.title('Classification Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

plt.subplot(2, 3, 5)
reg_scores = [mse_nn, pytorch_reg_mse]
plt.bar(frameworks, reg_scores, color=['blue', 'orange'], alpha=0.7)
plt.title('Regression MSE Comparison')
plt.ylabel('MSE')

plt.subplot(2, 3, 6)
with torch.no_grad():
    pytorch_reg_pred = pytorch_reg(torch.FloatTensor(X_reg_test)).numpy()
plt.scatter(y_reg_test, pytorch_reg_pred, alpha=0.6, color='orange')
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('PyTorch Regression: Predictions vs True')

plt.tight_layout()
plt.show()

print("🚀 Framework comparison complete!")

## 7. Quick Preview: Advanced Architectures 🎯

Let's get a taste of more advanced architectures that you'll master in the coming weeks!

In [ ]:
# Simple CNN for image-like data (preview)\nclass SimpleCNN(nn.Module):
    def __init__(self, input_channels=1, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # Assuming 28x28 input
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Simple RNN for sequence data (preview)
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # Take last output
        return out

# Simple Attention Mechanism (preview)
class SimpleAttention(nn.Module):
    def __init__(self, hidden_size):
        super(SimpleAttention, self).__init__()
        self.hidden_size = hidden_size
        self.attention = nn.Linear(hidden_size, 1)
    
    def forward(self, hidden_states):
        # hidden_states: (batch_size, seq_len, hidden_size)
        attention_weights = F.softmax(self.attention(hidden_states), dim=1)
        context = torch.sum(attention_weights * hidden_states, dim=1)
        return context, attention_weights

print("🎯 Advanced Architecture Previews Created!")
print("📚 You'll learn these in detail in upcoming weeks:")
print("   🖼️  CNNs for Computer Vision (Week 7-8)")
print("   🔄  RNNs for Sequences (Week 9)")  
print("   🎯  Attention & Transformers (Week 10-11)")

# Create sample data for demonstrations
sample_image = torch.randn(1, 1, 28, 28)  # Batch of 1, 1 channel, 28x28
sample_sequence = torch.randn(1, 10, 20)   # Batch of 1, sequence length 10, feature size 20
sample_hidden = torch.randn(1, 10, 64)     # For attention demo

# Test architectures
cnn = SimpleCNN(input_channels=1, num_classes=10)
rnn = SimpleRNN(input_size=20, hidden_size=64, output_size=5)
attention = SimpleAttention(hidden_size=64)

with torch.no_grad():
    cnn_output = cnn(sample_image)
    rnn_output = rnn(sample_sequence)
    context, attn_weights = attention(sample_hidden)

print(f"\\n🖼️  CNN output shape: {cnn_output.shape}")
print(f"🔄  RNN output shape: {rnn_output.shape}")
print(f"🎯  Attention context shape: {context.shape}")
print(f"🎯  Attention weights shape: {attn_weights.shape}")

print("\\n✨ These are just previews - you'll implement them fully later!")

## 8. Learning Resources & Next Steps 📚

Congratulations! You've completed your first deep learning notebook. Here's what you've learned and what comes next.

In [ ]:
# Summary of what you've learned
learning_summary = {
    "✅ Completed": [
        "Environment setup and library imports",
        "Data loading and preprocessing techniques", 
        "Linear regression from scratch and with PyTorch",
        "Neural networks from scratch (forward/backward propagation)",
        "Framework comparison (scratch vs PyTorch)",
        "Introduction to advanced architectures (CNN, RNN, Attention)"
    ],
    
    "🧠 Key Concepts Mastered": [
        "Gradient descent optimization",
        "Loss functions (MSE, Cross-entropy)",
        "Activation functions (ReLU, Sigmoid)",
        "Backpropagation algorithm",
        "Framework abstractions and benefits",
        "Model evaluation and visualization"
    ],
    
    "📚 d2l.ai Chapters to Read Next": [
        "Chapter 1: Introduction",
        "Chapter 2: Preliminaries", 
        "Chapter 3: Linear Neural Networks",
        "Chapter 4: Multilayer Perceptrons",
        "Chapter 5: Deep Learning Computation"
    ],
    
    "🎯 Next Week's Focus": [
        "Mathematical foundations (calculus, linear algebra)",
        "d2l.ai practical exercises",
        "More advanced optimization techniques",
        "Regularization methods",
        "Better data preprocessing"
    ]
}

print("🎉 CONGRATULATIONS! 🎉")
print("="*60)

for section, items in learning_summary.items():
    print(f"\\n{section}:")
    for item in items:
        print(f"  • {item}")

print("\\n" + "="*60)
print("🚀 READY FOR NEXT STEPS:")
print("   1. Review d2l.ai chapters 1-5")
print("   2. Complete exercises in 01_foundations/")
print("   3. Move to 02_ml_basics/ when ready")
print("   4. Join d2l.ai community discussions")
print("   5. Start your learning journal")

# Create a simple progress tracker
import json
import os

progress_data = {
    "completion_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook": "week1_deep_learning_mastery.ipynb",
    "status": "completed",
    "key_achievements": [
        "Built neural network from scratch",
        "Compared frameworks vs manual implementation", 
        "Understood backpropagation",
        "Previewed advanced architectures"
    ],
    "next_steps": [
        "Read d2l.ai chapters 1-5",
        "Complete math review",
        "Start week 2 materials"
    ]
}

# Save progress (optional)
progress_file = "../progress_tracker.json"
if os.path.exists(progress_file):
    with open(progress_file, 'r') as f:
        existing_progress = json.load(f)
    existing_progress.append(progress_data)
else:
    existing_progress = [progress_data]

try:
    with open(progress_file, 'w') as f:
        json.dump(existing_progress, f, indent=2)
    print(f"\\n📝 Progress saved to {progress_file}")
except:
    print(f"\\n📝 Note: Could not save progress file")

print("\\n🌟 Keep up the amazing work! Deep learning mastery is within reach! 🌟")